In [1]:
import pandas as pd
import os

BASE_PATH = ".."
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")

TRANS_PATH = os.path.join(
    PROCESSED_PATH,
    "transactions.csv"
)

trans_df = pd.read_csv(TRANS_PATH)

# Create account IDs
trans_df["Sender Account ID"] = (
    trans_df["From Bank"].astype(str)
    + "_"
    + trans_df["Account"].astype(str)
)

trans_df["Receiver Account ID"] = (
    trans_df["To Bank"].astype(str)
    + "_"
    + trans_df["Account.1"].astype(str)
)

# Convert amounts to numeric
trans_df["Amount Received"] = pd.to_numeric(
    trans_df["Amount Received"],
    errors="coerce"
)

trans_df["Amount Paid"] = pd.to_numeric(
    trans_df["Amount Paid"],
    errors="coerce"
)

print("Transactions:", trans_df.shape)

Transactions: (6924049, 14)


In [2]:
# Incoming behaviour for each receiver account
incoming = (
    trans_df.groupby("Receiver Account ID")
    .agg(
        Incoming_Transaction_Count=("Transaction ID", "count"),
        Total_Incoming_Amount=("Amount Received", "sum"),
        Average_Incoming_Amount=("Amount Received", "mean"),
        Unique_Senders=("Sender Account ID", "nunique")
    )
    .reset_index()
    .rename(columns={
        "Receiver Account ID": "Account ID"
    })
)

# Outgoing behaviour for each sender account
outgoing = (
    trans_df.groupby("Sender Account ID")
    .agg(
        Outgoing_Transaction_Count=("Transaction ID", "count"),
        Total_Outgoing_Amount=("Amount Paid", "sum"),
        Average_Outgoing_Amount=("Amount Paid", "mean"),
        Unique_Receivers=("Receiver Account ID", "nunique")
    )
    .reset_index()
    .rename(columns={
        "Sender Account ID": "Account ID"
    })
)

# Combine both sides
account_behaviour = incoming.merge(
    outgoing,
    on="Account ID",
    how="outer"
)

print(account_behaviour.head())
print("\nAccounts:", len(account_behaviour))

    Account ID  Incoming_Transaction_Count  Total_Incoming_Amount  \
0  0_800060CE0                         2.0               21117.61   
1  0_800061260                        21.0               81718.49   
2  0_800062D90                         2.0               15911.60   
3  0_800062F80                        35.0              166139.56   
4  0_800064980                         6.0              185448.70   

   Average_Incoming_Amount  Unique_Senders  Outgoing_Transaction_Count  \
0             10558.805000             1.0                       226.0   
1              3891.356667             2.0                       246.0   
2              7955.800000             2.0                       205.0   
3              4746.844571             2.0                       103.0   
4             30908.116667             4.0                       172.0   

   Total_Outgoing_Amount  Average_Outgoing_Amount  Unique_Receivers  
0            30475097.15            134845.562611              20.0  


In [3]:
# Sender-side behaviour
sender_behaviour = account_behaviour.rename(columns={
    "Account ID": "Sender Account ID",
    "Incoming_Transaction_Count": "Sender_Incoming_Count",
    "Outgoing_Transaction_Count": "Outgoing_Transaction_Count",
    "Total_Incoming_Amount": "Sender_Total_Incoming_Amount",
    "Total_Outgoing_Amount": "Total_Outgoing_Amount",
    "Average_Incoming_Amount": "Sender_Average_Incoming_Amount",
    "Average_Outgoing_Amount": "Average_Outgoing_Amount",
    "Unique_Senders": "Sender_Unique_Senders",
    "Unique_Receivers": "Unique_Receivers"
})

# Receiver-side behaviour
receiver_behaviour = account_behaviour.rename(columns={
    "Account ID": "Receiver Account ID",
    "Incoming_Transaction_Count": "Incoming_Transaction_Count",
    "Outgoing_Transaction_Count": "Receiver_Outgoing_Count",
    "Total_Incoming_Amount": "Total_Incoming_Amount",
    "Total_Outgoing_Amount": "Receiver_Total_Outgoing_Amount",
    "Average_Incoming_Amount": "Average_Incoming_Amount",
    "Average_Outgoing_Amount": "Receiver_Average_Outgoing_Amount",
    "Unique_Senders": "Unique_Senders",
    "Unique_Receivers": "Receiver_Unique_Receivers"
})

# Merge sender and receiver behaviour
behaviour_transactions = trans_df[
    [
        "Transaction ID",
        "Sender Account ID",
        "Receiver Account ID"
    ]
].copy()

behaviour_transactions = behaviour_transactions.merge(
    sender_behaviour[
        [
            "Sender Account ID",
            "Outgoing_Transaction_Count",
            "Total_Outgoing_Amount",
            "Average_Outgoing_Amount"
        ]
    ],
    on="Sender Account ID",
    how="left"
)

behaviour_transactions = behaviour_transactions.merge(
    receiver_behaviour[
        [
            "Receiver Account ID",
            "Incoming_Transaction_Count",
            "Total_Incoming_Amount",
            "Average_Incoming_Amount"
        ]
    ],
    on="Receiver Account ID",
    how="left"
)

# Unique counterparties
sender_unique = (
    trans_df.groupby("Sender Account ID")["Receiver Account ID"]
    .nunique()
    .rename("Unique Receivers")
)

receiver_unique = (
    trans_df.groupby("Receiver Account ID")["Sender Account ID"]
    .nunique()
    .rename("Unique Senders")
)

behaviour_transactions = behaviour_transactions.merge(
    sender_unique,
    on="Sender Account ID",
    how="left"
)

behaviour_transactions = behaviour_transactions.merge(
    receiver_unique,
    on="Receiver Account ID",
    how="left"
)

print(behaviour_transactions.head())

   Transaction ID Sender Account ID Receiver Account ID  \
0               1      11_8000ECA90        11_8000ECA90   
1               2    3402_80021DAD0      3402_80021DAD0   
2               3      11_8000ECA90      1120_8006AA910   
3               4    3814_8006AD080      3814_8006AD080   
4               5      20_8006AD530        20_8006AD530   

   Outgoing_Transaction_Count  Total_Outgoing_Amount  Average_Outgoing_Amount  \
0                        40.0            11991678.77            299791.969250   
1                         1.0                1858.96              1858.960000   
2                        40.0            11991678.77            299791.969250   
3                         1.0                  12.32                12.320000   
4                        42.0                7376.58               175.632857   

   Incoming_Transaction_Count  Total_Incoming_Amount  Average_Incoming_Amount  \
0                        40.0             3809722.14             95243.053500

In [4]:
account_behaviour_df = behaviour_transactions[
    [
        "Transaction ID",
        "Sender Account ID",
        "Receiver Account ID",
        "Incoming_Transaction_Count",
        "Outgoing_Transaction_Count",
        "Total_Incoming_Amount",
        "Total_Outgoing_Amount",
        "Average_Incoming_Amount",
        "Average_Outgoing_Amount",
        "Unique Senders",
        "Unique Receivers"
    ]
].copy()

account_behaviour_df.columns = [
    "Transaction ID",
    "Sender Account ID",
    "Receiver Account ID",
    "Incoming Transaction Count",
    "Outgoing Transaction Count",
    "Total Incoming Amount",
    "Total Outgoing Amount",
    "Average Incoming Amount",
    "Average Outgoing Amount",
    "Unique Senders",
    "Unique Receivers"
]

print("Shape:", account_behaviour_df.shape)

print("\nColumns:")
print(account_behaviour_df.columns.tolist())

print("\nMissing values:")
print(account_behaviour_df.isnull().sum())

# Missing behaviour means the account only appears on one side.
# Replace those values with 0.
numeric_columns = [
    "Incoming Transaction Count",
    "Outgoing Transaction Count",
    "Total Incoming Amount",
    "Total Outgoing Amount",
    "Average Incoming Amount",
    "Average Outgoing Amount",
    "Unique Senders",
    "Unique Receivers"
]

account_behaviour_df[numeric_columns] = (
    account_behaviour_df[numeric_columns].fillna(0)
)

OUTPUT_PATH = os.path.join(
    PROCESSED_PATH,
    "account_behaviour.csv"
)

account_behaviour_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:", OUTPUT_PATH)

Shape: (6924049, 11)

Columns:
['Transaction ID', 'Sender Account ID', 'Receiver Account ID', 'Incoming Transaction Count', 'Outgoing Transaction Count', 'Total Incoming Amount', 'Total Outgoing Amount', 'Average Incoming Amount', 'Average Outgoing Amount', 'Unique Senders', 'Unique Receivers']

Missing values:
Transaction ID                0
Sender Account ID             0
Receiver Account ID           0
Incoming Transaction Count    0
Outgoing Transaction Count    0
Total Incoming Amount         0
Total Outgoing Amount         0
Average Incoming Amount       0
Average Outgoing Amount       0
Unique Senders                0
Unique Receivers              0
dtype: int64

Saved: ..\data\processed\account_behaviour.csv
